# Chat loop and token budget (IKEA reviews)

Why the API forgets what you asked one call earlier, and what sending all IKEA reviews to a model costs: guess first, then measure.

In this notebook, we will:
1. Ask a follow-up question and see that the API forgets.
2. Keep a messages list and resend it.
3. Guess the cost of classifying all IKEA reviews.
4. Count tokens with `tiktoken` and compare them with the usage reported by the API.

**Time:** 10 minutes, together

## Libraries and settings

`ask()` = `client.responses.create()` plus a cache on disk: running a cell again is free. Code: `src/llm_course/client.py`.

In [ ]:
# Libraries
import os
import platform
import warnings
from datetime import datetime
from platform import python_version

import pandas as pd
import tiktoken

from llm_course import ask

# Ignore warnings
warnings.filterwarnings('ignore')

# Model and list price in USD per 1 million input tokens (31 Oct 2026)
MODEL = 'gpt-6-luna'
PRICE_INPUT_PER_M = 0.10

# Check that the API key is available (never print the key itself)
print('API key found:', 'OPENAI_API_KEY' in os.environ)

# Show current working directory
print(os.getcwd())

## Step 1: Ask the API a follow-up question

In [ ]:
# First question
r1 = ask(
    model=MODEL,
    input='Name one typical dish in the IKEA restaurant. Answer in one sentence.',
)
print('Answer 1:', r1.output_text)

# Follow-up question, sent as a separate call
r2 = ask(model=MODEL, input='Why is it so popular?')
print('Answer 2:', r2.output_text)

## Step 2: Keep a messages list and resend it

In [ ]:
# Stable rules go into the system message
RULES = 'You answer questions about IKEA. Keep every answer to one sentence.'

# The questions of our conversation
conversation = [
    'Name one typical dish in the IKEA restaurant.',
    'Why is it so popular?',
    'What was my first question?',
]

# Start the history with the system message
messages = [{'role': 'system', 'content': RULES}]

for i, question in enumerate(conversation, start=1):
    # Add the question, send the whole history, store the answer
    messages.append({'role': 'user', 'content': question})
    r = ask(model=MODEL, input=messages)
    messages.append({'role': 'assistant', 'content': r.output_text})

    print(f'Q{i}: {question}')
    print(f'A{i}: {r.output_text}')
    print(f'    input tokens of this call: {r.usage.input_tokens}', '\n')

Every API call is independent: the history must be resent, so each turn costs more input tokens. With `previous_response_id` the API keeps the history on the server, but earlier turns are still billed.

## Step 3: Guess the cost of classifying all IKEA reviews

Prompt per review: *"Positive, negative or mixed? Answer with one word."* Guess **before** you compute.

In [ ]:
# Your guess in USD (before you compute anything!)
my_guess_usd = None  # TODO: e.g. 0.50

## Import the IKEA reviews

In [ ]:
# Read all IKEA reviews (data from Moodle, see README: Data)
df = pd.read_csv(
    '../data/ikea_reviews.csv', sep=',', encoding='utf-8', parse_dates=['date']
)

# Shape and first rows
print(df.shape)
df[['store', 'city', 'stars', 'date', 'text']].head(5)

## Step 4: Count tokens with tiktoken

`o200k_base` is the tokenizer of GPT-4o and later OpenAI models.

In [ ]:
# Load the tokenizer
enc = tiktoken.get_encoding('o200k_base')

# A review from IKEA Philadelphia (1/5 stars, original spelling)
review = 'Swedish meatballs good. Attitude at.Return/Exchange counter bad.'

# Encode the text and decode every token separately
token_ids = enc.encode(review)
pieces = [enc.decode([t]) for t in token_ids]

print('Words: ', len(review.split()))
print('Tokens:', len(token_ids))
print(pieces)

**Your turn:** try a German sentence, a long word and a text with typos.

In [ ]:
# TODO: replace the text and run the cell again
my_text = 'Wir alle lieben IKEA, solange man von niemandem Hilfe braucht.'
print(len(enc.encode(my_text)), [enc.decode([t]) for t in enc.encode(my_text)])

## Count the tokens of all IKEA reviews

In [ ]:
# Number of tokens per review
df['tokens'] = df['text'].apply(lambda t: len(enc.encode(t)))

# Distribution of the review length in tokens
print(df['tokens'].describe().round(1))
print('Total tokens:', f"{df['tokens'].sum():,}")

## Compute the cost and compare it with your guess

In [ ]:
# The prompt is sent once per review
PROMPT = 'Positive, negative or mixed? Answer with one word.\n'
prompt_tokens = len(enc.encode(PROMPT))

# Input tokens of all calls and their cost
input_tokens = df['tokens'].sum() + len(df) * prompt_tokens
cost = input_tokens / 1_000_000 * PRICE_INPUT_PER_M

print(f'Input tokens:  {input_tokens:,}')
print(f'Cost (input):  {cost:.4f} USD')
print(f'Your guess:    {my_guess_usd} USD')

Output and reasoning tokens come on top.

## Compare tiktoken with the usage reported by the API

In [ ]:
# Send one review to the model
review = df.loc[0, 'text']
r = ask(model=MODEL, input=PROMPT + review)

print('Label:', r.output_text, '\n')
print('tiktoken count of the prompt:', len(enc.encode(PROMPT + review)))
print('Usage reported by the API:')
print(r.usage.model_dump_json(indent=2))

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
# Show operating system, date/time and Python version
print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')